# Quantiles and Quartiles

Describing data numerically is half the job — a data scientist also needs to *visualize* it correctly, and the language of most statistical plots is built on **quantiles**: the values that split your data into given proportions. The median splits it 50/50; the **first quartile** leaves a quarter to the left; and in general the **k% quantile** $q_{k/100}$ leaves $k\%$ below. This notebook builds the idea on a real advertising dataset and connects it to distributions via $P(X \le q_{k/100}) = \tfrac{k}{100}$. It is based on the twelfth video of Lesson 3 (Describing Distributions).

## 1. Introduction

Consider an advertising dataset: newspaper, TV, and radio ad budgets alongside the sales revenue they generated. For simplicity we take a 12-point sample of the **newspaper ad budget**:

$$
8.7,\ 14.2,\ 18.3,\ 18.4,\ 23.2,\ 25.9,\ 29.7,\ 35.2,\ 51.2,\ 54.7,\ 65.9,\ 75
$$

You already know one quantile by heart — the median. Quantiles generalize it from "split in half" to "split anywhere," and they are the foundation of the box plots, violin plots, and summary statistics you will meet next.

After completing this notebook you will be able to:

- Compute the **median** (50% quantile, $Q_2$), the **first quartile** $Q_1$ (25%), and the **third quartile** $Q_3$ (75%) of a dataset.
- Define and compute the general **k% quantile** $q_{k/100}$.
- Connect sample quantiles to distribution quantiles via $P(X \le q_{k/100}) = \tfrac{k}{100}$ — an area under the PDF.
- Be aware that software uses several interpolation conventions for sample quantiles.

## 2. Theory

### The median, revisited as a quantile

To find the median of the 12-point sample: **sort** the values in increasing order, split the dataset evenly into two halves of six samples each, and average the two values in the middle:

$$
\text{median} = \frac{25.9 + 29.7}{2} = 27.8
$$

We call this the **50% quantile**, or the **second quartile** $Q_2$: half the data lies to its left, half to its right.

### The first quartile

What about the point that leaves *one quarter* of the data to the left and *three quarters* to the right? With 12 points, a quarter is 3 points, so the split falls between the 3rd and 4th values:

$$
q_{0.25} = Q_1 = \frac{18.3 + 18.4}{2} = 18.35
$$

This is the **25% quantile**, or **first quartile** — like a median, but cutting at ¼ / ¾ instead of ½ / ½.

### The general k% quantile

For any percentage $k$: the **k% quantile** is the value that leaves $k\%$ of the data to the left and $(100-k)\%$ to the right, written $q_{k/100}$. The common ones have special names:

| quantile | notation | name |
|---|---|---|
| 25% | $q_{0.25}$ | first quartile $Q_1$ |
| 50% | $q_{0.5}$ | median, $Q_2$ |
| 75% | $q_{0.75}$ | third quartile $Q_3$ |

### From samples to distributions

Say your dataset has $n$ points and $n_k$ of them make up the first $k\%$ (e.g. $n_{20}$ is the first 20% of the sorted data). Then

$$
\frac{k}{100} = \frac{n_k}{n} \approx P\big(X \le q_{k/100}\big)
$$

— the fraction of data below the quantile estimates the *probability* of the variable being below it. If the data follows a distribution with a PDF, the picture is: the **area under the PDF to the left of $q_{k/100}$ equals $\tfrac{k}{100}$**. So for distributions, the definition becomes exact:

$$
q_{k/100} \text{ is the value such that } P\big(X \le q_{k/100}\big) = \frac{k}{100}
$$

(In other words, the quantile function is the inverse of the CDF.)

### A practical note: conventions differ

For samples, "the value with 25% below" often falls *between* two data points, and different software interpolates differently. The course's convention averages the two straddling values — `np.quantile(x, p, method="midpoint")` at the quartiles. NumPy's default (`method="linear"`) interpolates proportionally and can give slightly different numbers (e.g. 18.375 instead of 18.35 for $Q_1$ here). The concept is identical; only the tie-breaking differs.

## 3. Worked Examples

### Example 1: all three quartiles of the newspaper sample

Sorted data (n = 12): $8.7, 14.2, 18.3, \mid 18.4, 23.2, 25.9, \mid 29.7, 35.2, 51.2, \mid 54.7, 65.9, 75$

- $Q_1 = \frac{18.3 + 18.4}{2} = 18.35$ — a quarter of the budgets are below ≈ `$18k`.
- $Q_2 = \frac{25.9 + 29.7}{2} = 27.8$ — the median budget.
- $Q_3 = \frac{51.2 + 54.7}{2} = 52.95$ — a quarter of the budgets are *above* ≈ `$53k`.

### Example 2: a 90% quantile

$90\%$ of 12 points is $10.8$ points; the cut falls between the 10th (54.7) and 11th (65.9) sorted values. With the midpoint convention $q_{0.9} = \frac{54.7 + 65.9}{2} = 60.3$: only about 10% of newspaper budgets exceed `$60k`.

### Example 3: quantiles of a distribution

For the standard normal, $q_{0.5} = 0$ (symmetry) and $q_{0.75} \approx 0.674$ — the value with 75% of the bell's area to its left. (You met 0.674 before: $\pm 0.674\sigma$ brackets the central 50% — the interval from $q_{0.25}$ to $q_{0.75}$.) For the uniform distribution on $[0, 60]$, areas are proportional to length, so $q_{k/100} = 60 \cdot \tfrac{k}{100}$ exactly.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

newspaper = np.array([8.7, 14.2, 18.3, 18.4, 23.2, 25.9,
                      29.7, 35.2, 51.2, 54.7, 65.9, 75.0])

### Quartiles by hand and with numpy

In [ ]:
data = np.sort(newspaper)
n = len(data)

def midpoint_quantile(sorted_data, p):
    # the course's convention: if the cut falls between two points, average them
    return np.quantile(sorted_data, p, method="midpoint")

print(f"sorted data: {data}")
print()
print(f"Q1 (25% quantile): {midpoint_quantile(data, 0.25):.2f}   (= (18.3 + 18.4)/2)")
print(f"Q2 (median):       {midpoint_quantile(data, 0.50):.2f}   (= (25.9 + 29.7)/2)")
print(f"Q3 (75% quantile): {midpoint_quantile(data, 0.75):.2f}   (= (51.2 + 54.7)/2)")
print()
print(f"numpy default (linear interpolation) Q1: {np.quantile(data, 0.25)}")
print("Same idea, different tie-breaking convention -- know which one your tool uses.")

### The k% quantile for any k — and the fraction-below check

In [ ]:
for k in [10, 25, 50, 75, 90]:
    q = midpoint_quantile(newspaper, k / 100)
    fraction_below = np.mean(newspaper <= q)
    print(f"{k:>3}% quantile: q = {q:>6.2f}   fraction of data <= q: {fraction_below:.3f}")

### Quantiles of distributions: inverting the CDF

For a distribution, $q_{k/100}$ solves $P(X \le q) = k/100$. We find it numerically for the standard normal by integrating the PDF, and verify the uniform's exact formula.

In [ ]:
# Standard normal: build the CDF by numerical integration, then invert it
x = np.linspace(-5, 5, 100_001)
pdf = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)
cdf = np.cumsum(pdf) * (x[1] - x[0])

def normal_quantile(p):
    return x[np.searchsorted(cdf, p)]

print("standard normal quantiles:")
for p in [0.25, 0.5, 0.75, 0.9, 0.975]:
    print(f"  q_{p} = {normal_quantile(p):>7.3f}")
print("  (q_0.75 = 0.674: the 'probable error' from the standard-deviation notebook;")
print("   q_0.975 = 1.96: the classic 95%-interval endpoint)")

print("\nuniform on [0, 60]: q_k = 60 * k/100 exactly")
u = rng.uniform(0, 60, size=1_000_000)
for p in [0.25, 0.5, 0.9]:
    print(f"  q_{p}: sample = {np.quantile(u, p):>6.2f}, formula = {60 * p:>5.1f}")

### Sample quantiles approximate distribution quantiles

Draw growing samples from a normal distribution and watch the sample median and quartiles settle onto the true values.

In [ ]:
rng = np.random.default_rng(42)

mu, sigma = 27.8, 15
true_q = {0.25: mu - 0.6745 * sigma, 0.5: mu, 0.75: mu + 0.6745 * sigma}

print(f"{'n':>7} {'Q1':>8} {'median':>8} {'Q3':>8}")
for n in [12, 100, 1000, 100_000]:
    sample = rng.normal(mu, sigma, size=n)
    qs = np.quantile(sample, [0.25, 0.5, 0.75])
    print(f"{n:>7} {qs[0]:>8.2f} {qs[1]:>8.2f} {qs[2]:>8.2f}")
print(f"{'true':>7} {true_q[0.25]:>8.2f} {true_q[0.5]:>8.2f} {true_q[0.75]:>8.2f}")

## 5. Visualizations

### The newspaper sample and its quartiles

The sorted data as dots on a line, with the three quartiles cutting it into four groups of three.

In [ ]:
data = np.sort(newspaper)
q1, q2, q3 = [midpoint_quantile(data, p) for p in (0.25, 0.5, 0.75)]

plt.figure(figsize=(10, 3.5))
plt.plot(data, np.zeros_like(data), "o", markersize=10, color="tab:blue",
         label="newspaper budgets (sorted)")
for q, name, color in [(q1, "Q1 = 18.35", "tab:orange"),
                       (q2, "median = 27.8", "tab:red"),
                       (q3, "Q3 = 52.95", "tab:green")]:
    plt.axvline(q, color=color, linestyle="--", linewidth=2, label=name)
plt.title("Quartiles cut the sorted sample into four equal groups (3 points each)")
plt.xlabel("newspaper ad budget ($1000s)")
plt.yticks([])
plt.grid(True, alpha=0.3, axis="x")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

### The k% quantile as an area under the PDF

The video's closing picture: the shaded area to the left of $q_{0.2}$ is exactly 20% of the total.

In [ ]:
x = np.linspace(-4, 4, 1000)
pdf = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)
cdf_grid = np.cumsum(pdf) * (x[1] - x[0])
q20 = x[np.searchsorted(cdf_grid, 0.2)]

plt.figure(figsize=(9, 5))
plt.plot(x, pdf, color="tab:blue", linewidth=2, label="PDF")
plt.fill_between(x, pdf, where=x <= q20, color="tab:orange", alpha=0.6,
                 label="area = 20%")
plt.axvline(q20, color="black", linestyle=":", linewidth=2)
plt.annotate(f"$q_{{0.2}} = {q20:.2f}$\n$P(X \\leq q_{{0.2}}) = 0.2$",
             xy=(q20, 0.05), xytext=(q20 - 3.3, 0.22),
             arrowprops=dict(arrowstyle="->"), fontsize=11)
plt.title("The 20% quantile of a distribution: 20% of the area lies to its left")
plt.xlabel("value")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The quantile function is the inverse of the CDF

Left: the CDF maps values to cumulative probabilities. Right: the quantile function maps probabilities back to values — the same curve with its axes swapped.

In [ ]:
x = np.linspace(-3.5, 3.5, 2000)
pdf = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)
cdf_vals = np.cumsum(pdf) * (x[1] - x[0])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(x, cdf_vals, color="tab:blue", linewidth=2)
axes[0].set_title("CDF: value $\\to$ probability")
axes[0].set_xlabel("value q")
axes[0].set_ylabel("$P(X \\leq q)$")

axes[1].plot(cdf_vals, x, color="tab:green", linewidth=2)
axes[1].set_title("quantile function: probability $\\to$ value")
axes[1].set_xlabel("probability k/100")
axes[1].set_ylabel("$q_{k/100}$")

for p, name in [(0.25, "Q1"), (0.5, "median"), (0.75, "Q3")]:
    q = x[np.searchsorted(cdf_vals, p)]
    axes[0].plot([q, q, -3.5], [0, p, p], ":", color="gray", linewidth=1)
    axes[1].plot([p, p, 0], [-3.5, q, q], ":", color="gray", linewidth=1)
    axes[1].annotate(name, xy=(p, q), xytext=(p + 0.03, q - 0.55), fontsize=10)

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle("Quantiles invert the CDF (standard normal shown)")
plt.tight_layout()
plt.show()

### The whole story on one page

Everything above in a single sheet: the newspaper budgets cut into quartiles, a quantile as an area under the PDF and as the inverse CDF, and sample quartiles settling on the true ones.

<img src="img/12_quantiles.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Find the median, $Q_1$, and $Q_3$ of the 8-point dataset $[3, 7, 8, 5, 12, 14, 21, 13]$ by hand (sort first; use the midpoint convention), then check with `np.quantile(..., method="midpoint")`.

**Exercise 2.** For the newspaper sample, what is the 10% quantile? What does it say about the cheapest newspaper campaigns?

### Intermediate

**Exercise 3.** The **interquartile range** $\mathrm{IQR} = Q_3 - Q_1$ measures the spread of the middle half of the data (it will star in the next video's box plots). Compute the IQR of the newspaper sample, and explain in a sentence why the IQR — unlike the standard deviation — is unmoved when 75 is replaced by 750.

**Exercise 4.** For the exponential distribution with mean 5 (bus-waiting times), find the median from the CDF $P(X \le q) = 1 - e^{-q/5} = 0.5$ (solve for $q$), and verify by simulation. Why is the median *below* the mean here?

### Advanced

**Exercise 5.** The 68–95–99.7 rule in quantile language: for a normal distribution, which quantiles do $\mu - \sigma$ and $\mu + \sigma$ correspond to? Compute them numerically and express the rule as a statement about $q_{0.16}$ and $q_{0.84}$.

**Exercise 6.** **Inverse-transform sampling:** if $U$ is uniform on $[0,1]$, then $q_U$ (the quantile function applied to $U$) has exactly the target distribution. Use this to generate exponential(5) samples from uniform samples via $q_p = -5\ln(1-p)$, and compare the histogram with `rng.exponential(5, ...)`. Why must this work, given the definition of quantiles?

## 7. Solutions

In [ ]:
# Exercise 1: quartiles of a small dataset
small = np.array([3, 7, 8, 5, 12, 14, 21, 13])
sorted_small = np.sort(small)                 # [3 5 7 8 12 13 14 21]
print(f"Exercise 1: sorted = {sorted_small}")
print(f"median = (8 + 12)/2 = {np.quantile(small, 0.5, method='midpoint')}")
print(f"Q1 = (5 + 7)/2 = {np.quantile(small, 0.25, method='midpoint')}")
print(f"Q3 = (13 + 14)/2 = {np.quantile(small, 0.75, method='midpoint')}")

In [ ]:
# Exercise 2: the 10% quantile of the newspaper sample
q10 = np.quantile(newspaper, 0.10, method="midpoint")
print(f"Exercise 2: q_0.1 = {q10}")
print("The cheapest ~10% of newspaper campaigns spent less than about $16k --")
print("with only 12 points the cut falls between the 1st and 2nd values, so the")
print("midpoint convention lands on (14.2 + 18.3)/2 = 16.25.")

In [ ]:
# Exercise 3: the IQR and robustness
q1 = np.quantile(newspaper, 0.25, method="midpoint")
q3 = np.quantile(newspaper, 0.75, method="midpoint")
iqr = q3 - q1

corrupted = newspaper.copy()
corrupted[-1] = 750.0
q1c = np.quantile(corrupted, 0.25, method="midpoint")
q3c = np.quantile(corrupted, 0.75, method="midpoint")

print(f"Exercise 3: IQR = {q3} - {q1} = {iqr:.2f}")
print(f"With 75 -> 750: IQR = {q3c - q1c:.2f} (unchanged), but std jumps "
      f"{newspaper.std():.1f} -> {corrupted.std():.1f}")
print("Quartiles depend only on the ORDER of the data; how extreme the largest")
print("value is never enters -- like the median, the IQR is outlier-robust.")

In [ ]:
# Exercise 4: median of the exponential
rng = np.random.default_rng(0)

# 1 - exp(-q/5) = 0.5  =>  q = 5 ln 2
median_exact = 5 * np.log(2)
sim = rng.exponential(5, size=1_000_000)
print(f"Exercise 4: median = 5 ln(2) = {median_exact:.4f}, "
      f"simulated = {np.median(sim):.4f}, mean = {sim.mean():.3f}")
print("The right tail drags the MEAN up to 5, but the median only cares about")
print("the 50/50 split -- for right-skewed distributions median < mean.")

In [ ]:
# Exercise 5: the 68% rule as quantiles
x = np.linspace(-6, 6, 200_001)
pdf = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)
cdf = np.cumsum(pdf) * (x[1] - x[0])

p_below_minus1 = cdf[np.searchsorted(x, -1.0)]
p_below_plus1 = cdf[np.searchsorted(x, 1.0)]
print(f"Exercise 5: P(X <= mu - sigma) = {p_below_minus1:.3f}, "
      f"P(X <= mu + sigma) = {p_below_plus1:.3f}")
print("So mu - sigma is (about) the 16% quantile and mu + sigma the 84% quantile:")
print("q_0.16 to q_0.84 spans 84% - 16% = 68% of the distribution -- the rule restated.")

In [ ]:
# Exercise 6: inverse-transform sampling
rng = np.random.default_rng(0)

u = rng.random(500_000)
via_quantiles = -5 * np.log(1 - u)          # exponential quantile function
direct = rng.exponential(5, size=500_000)

plt.figure(figsize=(9, 4))
plt.hist(via_quantiles, bins=100, density=True, alpha=0.5, label="q(U): inverse transform")
plt.hist(direct, bins=100, density=True, alpha=0.5, label="rng.exponential")
plt.title("Inverse-transform sampling reproduces the target distribution")
plt.xlabel("value")
plt.ylabel("density")
plt.xlim(0, 40)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Exercise 6: it must work by definition: P(q_U <= q_p) = P(U <= p) = p,")
print("which is exactly the statement that q_p is the p-quantile of the output.")

## 8. Conclusion

- The **median** is the 50% quantile $Q_2$: sort, split in half, average the two middle values if needed (27.8 for the newspaper sample).
- The **k% quantile** $q_{k/100}$ leaves $k\%$ of the data to its left; the named ones are the quartiles $Q_1 = q_{0.25}$ (18.35 here), $Q_2 = q_{0.5}$, $Q_3 = q_{0.75}$ (52.95).
- For distributions the definition is exact: $q_{k/100}$ is the value with $P(X \le q_{k/100}) = \tfrac{k}{100}$ — an **area under the PDF**, making the quantile function the inverse of the CDF.
- Sample quantiles approximate distribution quantiles ($\tfrac{n_k}{n} \approx P(X \le q)$) and converge as $n$ grows; beware that software interpolation conventions differ slightly.
- Quantiles are order-based and therefore robust to outliers — the IQR $Q_3 - Q_1$ ignores how extreme the tails are.

Next: putting quantiles to work in **visualizations** — box plots and friends.